In [ ]:
import pandas as pd
import requests
import time
import os

print("🏈 Iniciando extracción CORREGIDA de Series Ofensivas (Drives)...")
temporada = 2026
lista_drives = []

for semana in range(1, 19):
    print(f"Buscando juegos de la Semana {semana}...")
    url_calendario = f"https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates={temporada}&seasontype=2&week={semana}"
    
    try:
        res = requests.get(url_calendario, timeout=10)
        if res.status_code == 200:
            for evento in res.json().get('events', []):
                id_partido = evento.get('id')
                
                url_summary = f"https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary?event={id_partido}"
                res_sum = requests.get(url_summary, timeout=10)
                
                if res_sum.status_code == 200:
                    datos_sum = res_sum.json()
                    
                    for drive in datos_sum.get('drives', {}).get('previous', []):
                        
                        # CORRECCIÓN 1: Extraer el ID directo y añadir Abreviación
                        equipo_data = drive.get('team', {})
                        
                        # CORRECCIÓN 2: Usar 'offensivePlays' con fallback a la lista de jugadas
                        jugadas_totales = drive.get('offensivePlays')
                        if not jugadas_totales:
                            jugadas_totales = len(drive.get('plays', []))
                            
                        lista_drives.append({
                            'ID_Partido': id_partido,
                            'Semana': semana,
                            'ID_Equipo_Ofensiva': equipo_data.get('id', 'N/A'),
                            'Abreviacion_Equipo': equipo_data.get('abbreviation', 'N/A'),
                            'Resultado_Serie': drive.get('displayResult', 'N/A'),
                            'Yardas_Avanzadas': drive.get('yards', 0),
                            'Jugadas_Totales': jugadas_totales,
                            'Tiempo_Posesion': drive.get('timeElapsed', {}).get('displayValue', '00:00'),
                            'Descripcion_Final': drive.get('description', 'N/A')
                        })
                time.sleep(1) # Pausa estratégica
    except Exception as e:
        print(f"Error en semana {semana}: {e}")

# Guardar en CSV
nombre_archivo = 'nfl_drives_2026.csv'
df_drives = pd.DataFrame(lista_drives)
df_drives.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(f"✅ ¡Completado! Se guardaron {len(df_drives)} series ofensivas en '{nombre_archivo}'.")

🏈 Iniciando extracción CORREGIDA de Series Ofensivas (Drives)...
Buscando juegos de la Semana 1...
Buscando juegos de la Semana 2...
Buscando juegos de la Semana 3...
